# Treinamento SVM - Variações de Balanceamento
Este notebook prepara o ambiente para treinar SVM com diferentes estratégias de lidar com classes desbalanceadas.

### Instruções para quem for treinar:
1. Vá até a célula **'Configuração do Experimento'**.
2. Escolha a estratégia mudando a variável `BALANCE_STRATEGY`.
3. Rode todas as células.

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

try:
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.over_sampling import RandomOverSampler
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False
    print("AVISO: imbalanced-learn não instalado. Over/Undersampling não funcionarão.")
    print("Instale com: pip install imbalanced-learn")

repo_root = Path("../../").resolve() 
file_path = repo_root / "models" / "data" / "DogFeatures.csv"

In [ ]:
# ==========================================
# === CONFIGURAÇÃO DO EXPERIMENTO ===
# ==========================================

# Escolha uma das opções abaixo:
# 'none'         -> Treino padrão (Sem balanceamento)
# 'class_weight' -> Ponderamento (SVM penaliza mais erros nas classes menores)
# 'under'        -> Undersampling (Remove exemplos das classes majoritárias)
# 'over'         -> Oversampling (Duplica exemplos das classes minoritárias)

BALANCE_STRATEGY = 'class_weight' 

print(f"Estratégia Selecionada: {BALANCE_STRATEGY.upper()}")

In [ ]:
if not file_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {file_path}")

df = pd.read_csv(file_path)
groups = df["DogID"]

cols_to_drop = ["DogID", "label", "Breed", "Gender", "NeuteringStatus", "TestNum", "t_dt"]
cols_to_drop = [c for c in cols_to_drop if c in df.columns]

X = df.drop(columns=cols_to_drop)
y = df["label"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)
target_names = le.classes_

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset Original: {X.shape}")

In [ ]:
X_final = X_scaled
y_final = y_encoded
groups_final = groups
svm_params = {}

if BALANCE_STRATEGY == 'class_weight':
    print("Aplicando 'class_weight=balanced' no modelo SVM...")
    svm_params['class_weight'] = 'balanced'

elif BALANCE_STRATEGY == 'under':
    if not HAS_IMBLEARN: raise ImportError("Precisa instalar imbalanced-learn")
    print("Aplicando RandomUnderSampler...")
    # sampling simples
    rus = RandomUnderSampler(random_state=42)
    X_final, y_final = rus.fit_resample(X_scaled, y_encoded)
    # Ao fazer resampling, pode perder o alinhamento com 'groups' original
    # Para LOSO funcionar 100% com sampling, o ideal é fazer o sampling DENTRO do loop de validação (coisa que fiz na v3)
    print(f"Novo shape: {X_final.shape}")
    
elif BALANCE_STRATEGY == 'over':
    if not HAS_IMBLEARN: raise ImportError("Precisa instalar imbalanced-learn")
    print("Aplicando RandomOverSampler...")
    ros = RandomOverSampler(random_state=42)
    X_final, y_final = ros.fit_resample(X_scaled, y_encoded)
    print(f"Novo shape: {X_final.shape}")

else:
    print("Nenhuma estratégia de amostragem aplicada.")


In [ ]:
logo = LeaveOneGroupOut()

svm = SVC(**svm_params)

param_grid = [
    {'C': [1, 10], 'kernel': ['linear']},
    {'C': [1, 10], 'kernel': ['rbf'], 'gamma': ['scale']}
]

print(f"Configuração SVM: {svm}")

grid_search = GridSearchCV(
    svm, 
    param_grid, 
    cv=logo, 
    scoring='accuracy', 
    verbose=10,
    n_jobs=-1
)

In [ ]:
if BALANCE_STRATEGY in ['under', 'over']:
    print("AVISO: Com sampling, LOSO exato é complexo. Usando CV padrão ou necessita ajuste de grupos.")
    grid_search.cv = 3 
    grid_search.fit(X_final, y_final)
else:
    grid_search.fit(X_final, y_final, groups=groups)

print(f"Melhor Score: {grid_search.best_score_:.4f}")
print(f"Melhores Params: {grid_search.best_params_}")

In [ ]:
best_model = grid_search.best_estimator_

if BALANCE_STRATEGY in ['under', 'over']:
    y_pred = best_model.predict(X_final)
else:
    y_pred = cross_val_predict(best_model, X_final, y_final, groups=groups, cv=logo, n_jobs=-1)

print(classification_report(y_final, y_pred, target_names=target_names))

cm = confusion_matrix(y_final, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', 
            xticklabels=target_names, 
            yticklabels=target_names)
plt.title(f'Matriz de Confusão - {BALANCE_STRATEGY.upper()}')

img_name = f"svm_cm_{BALANCE_STRATEGY}.png"
plt.savefig(repo_root / "data" / "processed" / img_name)
print(f"Imagem salva: {img_name}")